In [2]:
from ngsolve import *
from ngsolve.webgui import Draw
import numpy as np
import scipy.optimize
import csv

# class gel_3D:
#     def __init__(self, length=90.0, width =15.0, thickness=1.6, phi0=0.2, mu_bar=-0.3):
#         self.phi0 = phi0
#         self.mu_hat = mu_hat   # 🔥 NUEVO
        
#         self.entropic_unit = 136.6  # measured in MPa
#         self.G = 0.13               # measured in MPa
#         self.gamma = self.G/self.entropic_unit
#         self.chi =  0.348
#         self.density =  1.23 # measured in [g/mL]

#         self.L = length      # measured in mm
#         self.d = thickness    # measured in mm
#         self.w = width # measured in mm
        
#         def auxIsotropic(s):
#             return s*self.dH(s*s*s) + self.gamma
#         self.lambda_iso = scipy.optimize.fsolve(auxIsotropic, 1.7)[0]

#         def auxUniaxial(s):
#             return s*self.gamma + self.dH(s)
#         self.lambda_target = scipy.optimize.fsolve(auxUniaxial, 1.9)[0]
        
#         def auxEnergyDensity(lambda1, lambda2, lambda3):
#             gel=self; phi0=gel.phi0; G=gel.G; chi=gel.chi; nu=gel.entropic_unit
#             J= lambda1*lambda2*lambda3
#             phi = phi0/J
#             return 0.5*G*(lambda1**2 + lambda2**2 + lambda3**2) + nu*((J-phi0)*np.log(1-phi) + phi0*chi*(1-phi))

#         lambda_iso = self.lambda_iso
#         self.reference_energy_density = auxEnergyDensity(lambda_iso, lambda_iso, lambda_iso)

#     def phi(self, J):
#         return self.phi0/J

#     def H(self, J):
#         return (J - self.phi0)*log(1-self.phi(J))  + self.phi0 * self.chi*(1-self.phi(J))

#     def dH(self, J):
#         return self.phi(J) + np.log(1-self.phi(J)) + self.chi * self.phi(J)**2

#     def Gfun(self, lamb):
#         nu = self.entropic_unit
#         return (-self.dH(lamb)/lamb)*nu
        
#     # energy density in [MPa]
#     def W(self, F):
               
#         J = Det(F)
#         C_tensor = F.trans * F
        
#         gel = self
#         G = gel.G
#         nu = gel.entropic_unit
#         phi0 = gel.phi0
#         mu_hat = gel.mu_hat
        
#         reference_energy_density = self.reference_energy_density
        
#         # 🔥 concentración
#         C = (J - phi0) / nu
        
#         return 0.5*G*(Trace(C_tensor)) + nu*gel.H(J) - mu_hat*C - reference_energy_density

class gel_3D:
    def __init__(self, length=90.0, width =15.0, thickness=1.6, phi0=0.2, mu_bar=-0.3):
        self.phi0 = phi0
        print(f'Initial polymer volume fraction = {phi0:.2f} [dimensionless];', end=' ')
        self.mu_bar = mu_bar   # 🔥 NUEVO
        print(f'Normalized chemical potential = {mu_bar:.6f} [dimensionless?];', end=' ')
        #
        # Values for T and V_m taken from p.1584 in Kang & Huang, JMPS 58 (2010)
        T = 25+273.15   # 25ºC, in [K]
        K_B = 1.380649e-23  # in m^2*kg*s^{-2}*K^{-1}, i.e. in N*m/K
        V_m = 3e-29     # volume of a molecule of solvent, in this case water, in m^3
        # self.entropic_unit = 136.6  # measured in MPa
        self.entropic_unit = K_B*T/V_m*1e-6  # measured in MPa
        print(f'Entropic unit = {self.entropic_unit:.2f} [MPa];', end='\n')
        #
        # self.G = 0.13               # measured in MPa
        # self.gamma = self.G/self.entropic_unit
        self.gamma = 0.001           # As in the simulations of Sect. 5 in Kang & Huang JMPS 2010
        # self.chi =  0.348
        self.chi = 0.4
        self.G = self.gamma*self.entropic_unit
        print(f'gamma=N*V_m = {self.gamma:.2e} [dimensionless];', end=' ')
        print(f'Shear modulus = N*K_B*T = {self.G:.2f} [MPa];', end=' ')
        print(f'Flory parameter = {self.chi:.3f} [dimensionless];', end='\n')
        #
        vapor_pressure = 3.2e-3    # 3.2 KPa, but measured in MPa
        self.p0_bar = vapor_pressure/(self.entropic_unit)
        print(f'Normalized vapor pressure = {self.p0_bar:.2e} [dimensionless];', end=' ')
        self.p_bar = self.p0_bar * np.exp(mu_bar)
        print(f'External solvent pressure = {self.p_bar*self.entropic_unit*1e3:.2f} [KPa];', end=' ')
        print(f'Normalized external pressure = {self.p_bar:.2e} [MPa];', end='\n')
        #        
        # self.density =  1.23 # measured in [g/mL]

        self.L = length      # measured in mm
        self.d = thickness    # measured in mm
        self.w = width # measured in mm

        self.filename_suffix = f'_phi0={self.phi0:.1f}_muBarAbs={np.abs(mu_bar):.6f}'
        print(f'Filename suffix: ' + self.filename_suffix, end='\n')

        def auxIsotropic(s):
            return s*self.dH(s*s*s) + self.gamma
        max_attempts = 100000
        attempts = 0
        lambda_initial = phi0*1.1
        while attempts< max_attempts:
            aux_value = auxIsotropic(lambda_initial)
            if aux_value>0:
                break
            lambda_initial+=0.01
            attempts+=1
        self.lambda_iso = scipy.optimize.fsolve(auxIsotropic, lambda_initial)[0]
        print(f'Isotropic extension: {self.lambda_iso:.3f}; lambda_initial = {lambda_initial}', end=' ')

        def auxUniaxial(s):
            return s*self.gamma + self.dH(s)
        max_attempts = 100000
        attempts = 0
        lambda_initial = phi0*1.1
        while attempts< max_attempts:
            aux_value = auxUniaxial(lambda_initial)
            if aux_value>0:
                break
            lambda_initial+=0.01
            attempts+=1
        self.lambda_target = scipy.optimize.fsolve(auxUniaxial, lambda_initial)[0]
        print(f'Uniaxial extension: {self.lambda_target:.3f}; lambda_initial = {lambda_initial}', end='\n')

        def auxEnergyDensity(lambda1, lambda2, lambda3):
            gel=self; phi0=gel.phi0; G=gel.G; chi=gel.chi; nu=gel.entropic_unit; gamma=gel.gamma; mu_bar=gel.mu_bar; p_bar=gel.p_bar
            J= lambda1*lambda2*lambda3
            phi = phi0/J
            return 0.5*G*(lambda1**2 + lambda2**2 + lambda3**2 - 3) + nu*((J-phi0)*np.log(1-phi) + phi0*chi*(1-phi) - gamma*log(J) + (p_bar - mu_bar)*(J-phi0) )

        lambda_iso = self.lambda_iso
        self.reference_energy_density = auxEnergyDensity(lambda_iso, lambda_iso, lambda_iso)                
        print(f'Energy density of isotropic expansion: {self.reference_energy_density:.5f}', end=' ')

    def phi(self, J):
        return self.phi0/J

    def H(self, J):
        return (J - self.phi0)*log(1-self.phi(J))  + self.phi0 * self.chi*(1-self.phi(J)) - self.gamma*log(J) + (self.p_bar - self.mu_bar)*(J-self.phi0)

    def dH(self, J):
        return self.phi(J) + np.log(1-self.phi(J)) + self.chi * self.phi(J)**2  - self.gamma/J +self.p_bar - self.mu_bar

    def Gfun(self, lamb):
        nu = self.entropic_unit
        return (-self.dH(lamb)/lamb)*nu
    
    #Agregado Abril 01 de 2026 15:27
    def mu_fun(self, lamb):
        """
        Calcula el mu_bar asociado a una deformación uniaxial lambda.
        Basado en la condición de equilibrio uniaxial (tipo eq. 3.6 Kang & Huang 2010)
        """
        J = lamb        
        # return self.p_bar + self.gamma/lamb - phi - np.log(1 - phi) - self.chi * phi**2
        return self.phi(J)  + np.log(1-self.phi(J)) + self.chi * self.phi(J)**2 - self.gamma/J + self.gamma*lamb  + self.p0_bar
        
    # energy density in [MPa]
    def W(self, F):
               
        J = Det(F)
        C_tensor = F.trans * F
        
        gel = self
        G = gel.G
        nu = gel.entropic_unit               
        
        reference_energy_density =  self.reference_energy_density
        
        return 0.5*G*(Trace(C_tensor) - 3) + nu*gel.H(J) - reference_energy_density

In [3]:
### Main ###
# L = 90.0; d = 1.62; w = 15.0; phi0 = 0.2; order = 2; numIter = 0

data = ['', '90', '15.0', '1.62', '0.2', '0.0005'] 
# order = 1
order = 2
# numIteration = 4

L = float(data[1])
w = float(data[2])   # ✅ CORREGIDO
d = float(data[3])   # ✅ CORREGIDO
phi0 = float(data[4])
mu_bar = - float(data[5])

gel = gel_3D(length=L, width=w, thickness=d, phi0=phi0, mu_bar=mu_bar)

mesh_file = 'meshes/mesh0.vol.gz'
mesh = Mesh(mesh_file)

fes = VectorH1(mesh, order=order, dirichlet="bonded|debonded")
print('nDoF = {}'.format(fes.ndof))

nIterations = 15
indexes_iterations = range(nIterations+1)

for numIteration in indexes_iterations:
    # filename = f'gridfunctions/result_phi0=0.2_mesh0Polymer_order={order}_iter=' + str(numIter).zfill(2)
    # filename = f'gridfunctions/result_phi0=0.2__order={order}_iter=' + str(numIter).zfill(2)
    filename = 'gridfunctions/result' + \
        gel.filename_suffix + \
        "_order={}".format(order)
    filename +=  '_iter=' + str(numIteration).zfill(2)

    u = GridFunction(fes)
    u.Load(filename  + '.gfu')

    I = Id(mesh.dim)
    F = I + Grad(u)

    GF_energy_density = GridFunction(H1(mesh, order=1))
    GF_energy_density.Set(gel.W(F))

    Draw(GF_energy_density*1e3, mesh, deformation=u, min=0.0, max=152.0)

    elasticEnergy = Integrate(GF_energy_density, mesh, order=5)
    print('Total energy [mJ]: {:.2f}'.format(elasticEnergy))    

    vtk = VTKOutput(
        mesh,
        coefs=[u, GF_energy_density*1e3],
        names=["u", "Energy density [kPa]"],
        filename=filename,
        subdivision=1
    )
    vtk.Do()

Initial polymer volume fraction = 0.20 [dimensionless]; Normalized chemical potential = -0.000500 [dimensionless?]; Entropic unit = 137.21 [MPa];
gamma=N*V_m = 1.00e-03 [dimensionless]; Shear modulus = N*K_B*T = 0.14 [MPa]; Flory parameter = 0.400 [dimensionless];
Normalized vapor pressure = 2.33e-05 [dimensionless]; External solvent pressure = 3.20 [KPa]; Normalized external pressure = 2.33e-05 [MPa];
Filename suffix: _phi0=0.2_muBarAbs=0.000500
Isotropic extension: 1.349; lambda_initial = 1.350000000000001 Uniaxial extension: 1.790; lambda_initial = 1.8000000000000014
Energy density of isotropic expansion: -16.00297 nDoF = 586314


/tmp/ipykernel_20746/3837620338.py:157: RuntimeWarning: invalid value encountered in log
  return self.phi(J) + np.log(1-self.phi(J)) + self.chi * self.phi(J)**2  - self.gamma/J +self.p_bar - self.mu_bar


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

Total energy [mJ]: 585.48


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

...

Total energy [mJ]: nan


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

Total energy [mJ]: nan


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

Total energy [mJ]: nan
